In [1]:
import tensorflow as tf
import numpy as np
import os as os
tfkl = tf.keras.layers
import random
import sys                                                                                                                          

In [8]:
import training_functions_jo as training_functions
import importlib

importlib.reload(training_functions)

<module 'training_functions_jo' from '/Users/paigepark/Desktop/repos/deep-mort/code_2026/tuning/training_functions_jo.py'>

In [4]:
country_training = np.loadtxt('../../data/country_training.txt')
country_test = np.loadtxt('../../data/country_test.txt')

In [5]:
# prep data
country_train_prepped = training_functions.prep_data(country_training, mode="train", changeratetolog=True)
country_test_prepped = training_functions.prep_data(country_test, mode="test", changeratetolog=True)

In [6]:
# get the proper geography input dimension for model set up 
unique_vals = tf.unique(country_training[:, 0]).y
country_geo_dim = np.array(tf.size(unique_vals)).item()
country_geo_dim = country_geo_dim + 50

In [36]:
# # run all country model
# for i in range(1,6):
# Set reproducible seeds per iteration
np.random.seed(43)
tf.random.set_seed(43)
random.seed(43)
os.environ['PYTHONHASHSEED'] = str(43)

model_country, loss_info_country = training_functions.run_deep_model(country_train_prepped, country_test_prepped, country_geo_dim, 
                                                                        epochs=20, 
                                                                        steps_per_epoch=1405, 
                                                                        lograte=True)

training_input_features = (tf.convert_to_tensor((country_training[:,2] - 1959) / 60, dtype=tf.float32),  # Normalized year
                        tf.convert_to_tensor(country_training[:,3], dtype=tf.float32),  # Age
                        tf.convert_to_tensor(country_training[:,0], dtype=tf.float32),  # Geography
                        tf.convert_to_tensor(country_training[:,1], dtype=tf.float32))  # Gender

test_input_features = (tf.convert_to_tensor((country_test[:,2] - 1959) / 60, dtype=tf.float32),  # Normalized year
                    tf.convert_to_tensor(country_test[:,3], dtype=tf.float32),  # Age
                    tf.convert_to_tensor(country_test[:,0], dtype=tf.float32),  # Geography
                    tf.convert_to_tensor(country_test[:,1], dtype=tf.float32))  # Gender

training_predictions = model_country.predict(training_input_features)

test_predictions = model_country.predict(test_input_features)

inputs = np.delete(country_training, 4, axis=1)
training_predictions = np.column_stack((inputs, training_predictions))
inputs_test = np.delete(country_test, 4, axis=1)
test_predictions = np.column_stack((inputs_test, test_predictions))

Epoch 1/20
1405/1405 - 6s - 4ms/step - loss: 1.9191 - val_loss: 0.3042 - learning_rate: 1.0000e-03
Epoch 2/20
1405/1405 - 4s - 3ms/step - loss: 0.3378 - val_loss: 0.1937 - learning_rate: 1.0000e-03
Epoch 3/20
1405/1405 - 4s - 3ms/step - loss: 0.2512 - val_loss: 0.1891 - learning_rate: 1.0000e-03
Epoch 4/20
1405/1405 - 4s - 3ms/step - loss: 0.2168 - val_loss: 0.1659 - learning_rate: 1.0000e-03
Epoch 5/20
1405/1405 - 4s - 3ms/step - loss: 0.2060 - val_loss: 0.1610 - learning_rate: 1.0000e-03
Epoch 6/20
1405/1405 - 4s - 3ms/step - loss: 0.1875 - val_loss: 0.1684 - learning_rate: 1.0000e-03
Epoch 7/20
1405/1405 - 4s - 3ms/step - loss: 0.1820 - val_loss: 0.1513 - learning_rate: 1.0000e-03
Epoch 8/20
1405/1405 - 5s - 3ms/step - loss: 0.1811 - val_loss: 0.1881 - learning_rate: 1.0000e-03
Epoch 9/20
1405/1405 - 4s - 3ms/step - loss: 0.1702 - val_loss: 0.1771 - learning_rate: 1.0000e-03
Epoch 10/20
1405/1405 - 4s - 3ms/step - loss: 0.1688 - val_loss: 0.1524 - learning_rate: 1.0000e-03
Epoch 11/

In [45]:
train_rse = np.abs(training_predictions[:, -1] - np.log(np.maximum(country_training[:, 4], 9e-6)))
test_rse = np.abs(test_predictions[:, -1] - np.log(np.maximum(country_test[:, 4], 9e-6)))

inputs = np.delete(country_training, 4, axis=1)
train_rse = np.column_stack((inputs, train_rse))
inputs_test = np.delete(country_test, 4, axis=1)
test_rse = np.column_stack((inputs_test, test_rse))

prepped_train_rse = training_functions.prep_data(train_rse, mode="train", changeratetolog=False)
prepped_test_rse = training_functions.prep_data(test_rse, mode="train", changeratetolog=False)

In [40]:
meansd = np.mean(train_rse[:, -1])
train_coverage = np.mean(train_rse[:, -1] <= meansd * 1.96)
print(f"Training coverage: {train_coverage:.4f}")

Training coverage: 0.8981


In [41]:
coherent_forecast = np.genfromtxt('../../data/tuning_coherent_forecast.txt')
coherent_fitted = np.genfromtxt('../../data/tuning_coherent_fitted.txt')

In [43]:
print(coherent_forecast[:10, 8:])

[[3.32172028e-03 5.13039171e-03]
 [2.80488806e-04 5.18250220e-04]
 [1.57475218e-04 3.34548556e-04]
 [1.11622483e-04 2.93505154e-04]
 [9.47043324e-05 2.49813498e-04]
 [7.89636939e-05 1.75490876e-04]
 [6.47291814e-05 1.96395082e-04]
 [4.99898833e-05 1.85619768e-04]
 [5.87421572e-05 1.48540357e-04]
 [5.13466812e-05 1.32327721e-04]]


In [46]:

# Set reproducible seeds per iteration
np.random.seed(43)
tf.random.set_seed(43)
random.seed(43)
os.environ['PYTHONHASHSEED'] = str(43)

model_sd_country, loss_info_country = training_functions.run_deep_model(prepped_train_rse, prepped_test_rse, country_geo_dim, 
                                                                        epochs=20, 
                                                                        steps_per_epoch=1405, 
                                                                        lograte=False)

training_input_features = (tf.convert_to_tensor((country_training[:,2] - 1959) / 60, dtype=tf.float32),  # Normalized year
                        tf.convert_to_tensor(country_training[:,3], dtype=tf.float32),  # Age
                        tf.convert_to_tensor(country_training[:,0], dtype=tf.float32),  # Geography
                        tf.convert_to_tensor(country_training[:,1], dtype=tf.float32))  # Gender

test_input_features = (tf.convert_to_tensor((country_test[:,2] - 1959) / 60, dtype=tf.float32),  # Normalized year
                    tf.convert_to_tensor(country_test[:,3], dtype=tf.float32),  # Age
                    tf.convert_to_tensor(country_test[:,0], dtype=tf.float32),  # Geography
                    tf.convert_to_tensor(country_test[:,1], dtype=tf.float32))  # Gender

training_sd_predictions = model_sd_country.predict(training_input_features)

test_sd_predictions = model_sd_country.predict(test_input_features)

inputs = np.delete(country_training, 4, axis=1)
training_sd_predictions = np.column_stack((inputs, training_sd_predictions))
inputs_test = np.delete(country_test, 4, axis=1)
test_sd_predictions = np.column_stack((inputs_test, test_sd_predictions))

Epoch 1/20
1405/1405 - 6s - 4ms/step - loss: 0.0667 - val_loss: 0.0771 - learning_rate: 1.0000e-03
Epoch 2/20
1405/1405 - 5s - 3ms/step - loss: 0.0610 - val_loss: 0.0735 - learning_rate: 1.0000e-03
Epoch 3/20
1405/1405 - 5s - 3ms/step - loss: 0.0563 - val_loss: 0.0761 - learning_rate: 1.0000e-03
Epoch 4/20
1405/1405 - 4s - 3ms/step - loss: 0.0551 - val_loss: 0.0786 - learning_rate: 1.0000e-03
Epoch 5/20
1405/1405 - 4s - 3ms/step - loss: 0.0574 - val_loss: 0.0722 - learning_rate: 1.0000e-03
Epoch 6/20
1405/1405 - 5s - 3ms/step - loss: 0.0571 - val_loss: 0.0808 - learning_rate: 1.0000e-03
Epoch 7/20
1405/1405 - 5s - 3ms/step - loss: 0.0559 - val_loss: 0.0736 - learning_rate: 1.0000e-03
Epoch 8/20
1405/1405 - 5s - 3ms/step - loss: 0.0573 - val_loss: 0.0747 - learning_rate: 1.0000e-03
Epoch 9/20
1405/1405 - 5s - 3ms/step - loss: 0.0581 - val_loss: 0.0712 - learning_rate: 2.5000e-04
Epoch 10/20
1405/1405 - 5s - 4ms/step - loss: 0.0551 - val_loss: 0.0732 - learning_rate: 2.5000e-04
Epoch 11/

In [48]:
print(np.mean(test_rse[:, -1] <= test_sd_predictions[:, -1] * 1.96))

0.7110555555555556
